# Papermill Sequencer (Wave Runner)

In [1]:
import os, json
import papermill as pm
import pathlib as pl
from IPython.display import clear_output

In [2]:
import datetime
import time
import shutil
import glob
import json
import re

In [3]:
wave = 12

In [4]:
#set working directory and folder variables
os.chdir('..')

In [5]:
home = pl.Path(os.getcwd())
print('home is at ',home)

inputs = home/'inputs'
outputs_base = home/'outputs'
notebook_base = home/'notebooks'
assert home.stem == '_code', "Home should be at _code. You may be in the wrong folder. Try restarting kernal and reruning."

home is at  U:\173432208011\studies\1_great_divide_green_watershed\production\engineering_riverine\hydraulics\hydrology_incorporation\_code


In [6]:
#user to set main project name for the project
project = 'wy_fy22'

In [7]:
sequencer = {1: ['1404010610', '1404020005', '1404020008', '1404020007'], 2: ['1404010609', '1404010903', '1404020004'], 3: ['1404010607', '1404010608'], 4: ['1404010603', '1404010604', '1404010605'], 5: ['1404010601', '1404010602', '1404010710'], 6: ['1404010306', '1404010509', '1404010706', '1404010707', '1404010708', '1404010709'], 7: ['1404010113', '1404010301', '1404010303', '1404010304', '1404010305', '1404010406', '1404010503', '1404010504', '1404010506', '1404010507', '1404010508', '1404010702', '1404010703', '1404010704', '1404010705', '1404010803'], 8: ['1404010111', '1404010112', '1404010302', '1404010401', '1404010403', '1404010404', '1404010502', '1404010701', '1404010802'], 9: ['1404010109', '1404010110', '1404010402', '1404010801'], 10: ['1404010106', '1404010107', '1404010108'], 11: ['1404010102', '1404010103', '1404010104', '1404010105', '1404010206'], 12: ['1404010201', '1404010202', '1404010203', '1404010204', '1404010205']}

In [8]:
#get ds connections
with open(inputs/project/'dictionaries'/'HUC10_outflow_toHUC10.json') as src:
    huc_connect_huc = json.load(src)

In [9]:
#set variables
hucs = sequencer[wave]
#restart date variable setting
restart_date = '16/12/2024'
restart_ts = time.mktime(datetime.datetime.strptime(restart_date, "%d/%m/%Y").timetuple())

In [10]:
#############################

In [11]:
#########
hucs = ['1404010608']
#########

In [12]:
#############################

In [13]:
if not os.path.exists(outputs_base/'notebook_outputs'):
    os.makedirs(outputs_base/'notebook_outputs')

### Huc data creation


In [15]:
ds_hucs = []
for huc in hucs:
    print(huc)
    huc=huc
    project= project
    home = str(home) 
    res = pm.execute_notebook(notebook_base/'auto_bc_creation_WY_MG2.ipynb',outputs_base/'notebook_outputs'/f'auto_bc_creation_WY_{huc}.ipynb',
                              parameters= dict(huc=huc,project=project,home=home))
    res2 = pm.execute_notebook(notebook_base/'us_event_transfer_ds.ipynb',outputs_base/'notebook_outputs'/f'us_event_transfer_ds{huc}.ipynb',
                              parameters= dict(target=huc,project=project,home=home))
    if huc_connect_huc[huc] not in ds_hucs:
        ds_hucs.append(huc_connect_huc[huc])


1404010608


### Apply ds boundary condition


In [17]:
run_hucs = os.listdir(outputs_base/project/'trial_us_to_ds_events')
for huc in ds_hucs:
    for key, val in huc_connect_huc.items():
        #check that us_event transfer has been run for all us hucs
        if val == huc:
            assert f'wy_gdg_{key}' in run_hucs,f"Not all hucs upstream of {huc} have been run through the new process."
    #check that cloud output exists and is newer than 12/16
    assert os.path.exists(inputs/project/'cloud_output'/huc),'Download cloud output to the inputs/cloud_output folder'
    cloud_fol_mod_ts = os.path.getmtime(inputs/project/'cloud_output'/huc)
    assert cloud_fol_mod_ts > restart_ts, "cloud_output does not appear to have been updated since the restart process. Please check."
    res3 = pm.execute_notebook(notebook_base/'cross_bc_alignment_WY_MG10.ipynb',outputs_base/'notebook_outputs'/f'cross_bc_alignment_WY{huc}.ipynb',
                            parameters= dict(huc=huc,project=project,home=home))

Executing:   0%|          | 0/36 [00:00<?, ?cell/s]

### Copy and move files


In [18]:
home = pl.Path(home)

In [19]:
print(home.parent)

U:\173432208011\studies\1_great_divide_green_watershed\production\engineering_riverine\hydraulics\hydrology_incorporation


In [20]:
hucs

['1404010608']

In [21]:
#copy all files from eb_mod to 4_v5 folder
for huc in hucs:
    print(huc)
    eb_out = outputs_base/project/'eb_mod'/f'wy_gdg_{huc}'
    ready_for_cloud = home.parent/'4_v5_ready_for_cloud'/huc
    post_itr = home.parent/'6_cloud_qc_from_dewberry'/'revised_models'/huc

    #copy post-ITR files
    shutil.copytree(str(post_itr),str(ready_for_cloud))
    
    #check that eb mod output exists and is newer than 12/16
    assert os.path.exists(outputs_base/project/'eb_mod'/f'wy_gdg_{huc}'),'Does eb merge need to be run????'
    eb_fol_mod_ts = os.path.getmtime(outputs_base/project/'eb_mod'/f'wy_gdg_{huc}')
    assert eb_fol_mod_ts > restart_ts, "eb_mod output does not appeak to have been updated since the restart process. Please check."
    #DO NOT COPY OVER GEOMETRY FILE!!!
    geom_files = glob.glob(str(eb_out/'*.g*'))
    for g in geom_files:
        # os.path.unlink(str(eb_out/g))
        ##########
        os.unlink(str(eb_out/g))
        #########
    shutil.copytree(str(eb_out),str(ready_for_cloud),dirs_exist_ok = True )

1404010608


### Update prj file

In [22]:
for huc in hucs:
    print(huc)
    ready_for_cloud = home.parent/'4_v5_ready_for_cloud'/huc
    eb_out = outputs_base/project/'eb_mod'/f'wy_gdg_{huc}'
    prj_out_name = ready_for_cloud/f'wy_gdg_{huc}.prj'
    assert os.path.exists(prj_out_name),f'Project file (.prj) in ready for cloud folder does not exist for {huc}'
    new_desc = '\n'.join([f'HUC Number: {huc}',
    'Project Name - Wyoming - The Upper Great Green Divide',
    'MIP Number: 23-08-0020S',
    'Task Order Number: 70FBR822F00000027',
    'Company/city: STARR II / Calverton, MD',
    'Client: FEMA Region 8',
    'Brief Model Description: This is a BLE Level A HEC-RAS model. The model utilized inflow hydrographs generated as output from SST procedure. The analysis includes the 10%, 4%, 2%, 1%, and 0.2% annual-chance events and 1% plus/minus events.',
    'Model Version: HEC-RAS v 6.4.1',
    'Study Date: December 2024',
    'Topographic Data Source/Date: 1 meter (Southwest 2020)',
    'Vertical Datum: NAVD88',
    'Horizontal Datum: GCS_North_American_1983',
    'Geographical Coordinate System: USA_Contiguous_Albers_Equal_Area_Conic_USGS_version'])
    #additional information
    with open(str(inputs/project/'dictionaries'/'completed_event_dictionary.json'),'r') as ed:
        e_dict = json.load(ed)
    event_info = 'Recurrence interval / events information: \n '+str(e_dict[huc]).replace('],','],\n')[1:-1]
    no_outlet_info = ''
    if huc_connect_huc[huc] == 'N/A':
        no_outlet_info+='Please note that this is a closed basin and does not have any outflow boundary conditions'
    updated_dec = new_desc+'\n'+event_info+'\n'+ no_outlet_info
    #update project file
    with open(prj_out_name, "r+",encoding="ISO-8859-1") as f:
        file_contents = f.read()
        finder_desc = '(?<=BEGIN DESCRIPTION:\\n)[\s\S]+(?=\\nEND DESCRIPTION:[ \n ]DSS)'
        p_replace = re.search(finder_desc,file_contents)
        new_prj = file_contents[:p_replace.span()[0]]+updated_dec+file_contents[p_replace.span()[1]:]
        f.seek(0)
        f.write(new_prj)

1404010608


### "Print confirmation of completion"

In [24]:
print(hucs, 'are ready to be run in the cloud after folder pruning. \n Once cloud run is completed, please download the project dss file, geometry files, plan files, flow files, to _code/inputs/cloud_outputs before running the next wave')

['1404010608'] are ready to be run in the cloud after folder pruning. 
 Once cloud run is completed, please download the project dss file, geometry files, plan files, flow files, to _code/inputs/cloud_outputs before running the next wave
